---
2.2 快速上手参考答案
---

1. （单选题）基于 npugraph_ex 快速上手相关知识，下列说法错误的是？

    答案：C

   解析：npugraph_ex 仅面向在线推理，暂不支持反向流程 Capture 成图，不适用训练场景。A/B/D 均为文档原文约束。

2. （单选题）使用 torch.compile 开启 npugraph_ex 图模式，必须指定的 backend 参数是？

    答案：C

    解析：backend 参数用于选择编译后端，昇腾图捕获专用后端名称固定为 npugraph_ex；inductor 是通用 CPU/GPU 后端，无 npu、ascend 合法后端名。

3. （单选题）开启 npugraph_ex 整图捕获、禁止图中断，需要配置哪个参数为 True？

    答案：B

    解析：fullgraph=True 强制完整捕获模型整张计算图，遇到无法追踪代码直接报错，避免图碎片化；dynamic 控制动态维度，disable 关闭编译，mode 为自动调优参数，和整图捕获无关。

4. （单选题）关于 npugraph_ex 首次推理与二次推理耗时差异，下列说法正确的是？

    答案：C

    解析：第一次推理执行 Capture，收集算子、缓存任务流，存在大量初始化开销；后续推理直接复用缓存任务执行 Replay，无需重复捕获，时延大幅降低


2. （实验题）请基于本章 SimpleModel 示例代码完成改造

Eager模式代码示例：

In [ ]:
import torch
import torch_npu

# ===================== 环境校验 =====================
print("===== 环境校验信息 =====")
print(f"PyTorch version: {torch.__version__}")
print(f"torch_npu version: {torch_npu.__version__}")
print(f"NPU可用状态: {torch_npu.npu.is_available()}")
print(f"NPU设备数量: {torch_npu.npu.device_count()}")
if torch_npu.npu.is_available():
    print(f"0号NPU设备名: {torch_npu.npu.get_device_name(0)}")

# ===================== 定义模型 =====================
class SimpleModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, y):
        h1 = torch.add(x, y)
        h2 = torch.add(h1, x)
        h3 = torch.add(h2, y)
        h4 = torch.add(h3, y)
        h5 = torch.add(h4, y)
        return h5

# 模型迁移至NPU
model = SimpleModel().npu()

# 构造固定输入张量
x = torch.randn(2, 64).npu()
y = torch.randn(2, 64).npu()

out = model(x, y)
